In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
Whisper encoder-only audio feature extraction.
Loads audio via torchaudio, runs through Whisper-large-v3 encoder,
mean-pools over time → [1280] float32 tensor.
Saves → audio_features/train/{id_EXIST}.pt
"""

import torch
import pandas as pd
import numpy as np
import whisper
import torchaudio
import torchaudio.transforms as T
from pathlib import Path
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
VIDEO_DIR   = Path("./Dataset/test/videos/")
OUTPUT_DIR  = Path("./audio_features/test/")
JSON_PATH   = "./Dataset/test/EXIST2026_test_clean.json"
WHISPER_MODEL = "large-v3"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR   = 16000
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading Whisper {WHISPER_MODEL} on {DEVICE}...")
model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
model.eval()

df = pd.read_json(JSON_PATH).transpose()

resampler_cache = {}  # cache resamplers by original SR

def load_audio_mono_16k(video_path: Path) -> torch.Tensor | None:
    """
    Load audio from video file via torchaudio.
    Returns [T] float32 mono waveform at 16kHz.
    """
    try:
        waveform, sr = torchaudio.load(str(video_path))   # [C, T]
        # mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)  # [1, T]
        else:
            waveform = waveform                             # [1, T]
        # resample if needed
        if sr != TARGET_SR:
            if sr not in resampler_cache:
                resampler_cache[sr] = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
            waveform = resampler_cache[sr](waveform)
        return waveform.squeeze(0)   # [T]
    except Exception as e:
        print(f"  [ERR] torchaudio.load failed: {e}")
        return None


@torch.no_grad()
def extract_whisper_features(waveform: torch.Tensor) -> torch.Tensor:
    """
    Run waveform through Whisper encoder only.
    Input:  [T] float32 at 16kHz
    Output: [1280] mean-pooled encoder features
    """
    # Whisper expects exactly 30s window
    audio = whisper.pad_or_trim(waveform.numpy())          # numpy [480000]
    mel   = whisper.log_mel_spectrogram(audio,
                                        n_mels=model.dims.n_mels).to(DEVICE)  # [80/128, 3000]
    mel   = mel.unsqueeze(0)                               # [1, n_mels, 3000]

    encoder_out = model.encoder(mel)                       # [1, 1500, 1280]
    features    = encoder_out.squeeze(0).mean(dim=0)       # [1280]
    return features.cpu().float()


skipped = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting audio"):
    item_id  = str(row["id_EXIST"])
    out_path = OUTPUT_DIR / f"{item_id}.pt"

    if out_path.exists():
        continue  # resume-safe

    video_path = VIDEO_DIR / Path(row["path_video"]).name
    if not video_path.exists():
        print(f"[WARN] Video not found: {item_id}")
        skipped.append(item_id)
        torch.save(torch.zeros(1280), out_path)
        continue

    waveform = load_audio_mono_16k(video_path)
    if waveform is None:
        skipped.append(item_id)
        torch.save(torch.zeros(1280), out_path)
        continue

    features = extract_whisper_features(waveform)          # [1280]
    torch.save(features, out_path)

print(f"\nDone. Skipped {len(skipped)} videos.")
if skipped:
    print("Skipped IDs:", skipped[:20])

Loading Whisper large-v3 on cuda...


Extracting audio: 100%|███████████████████████████| 674/674 [01:29<00:00,  7.50it/s]


Done. Skipped 0 videos.
